# WaveForge — Multi-GPU Benchmark on Kaggle

**Kaggle advantages over Colab:**
- **2× T4 GPUs** (30 hours/week free) or **P100** (30 hrs/week)
- **20 GB RAM** (vs 12 GB Colab)
- **No timeout disconnects** — runs complete even if you close the browser
- **Persistent `/kaggle/working/` storage** — results never lost
- **Internet enabled** — can push to GitHub automatically

**What this notebook does:**
1. Detects all available GPUs
2. Runs WaveForge benchmarks across all GPU sizes
3. Runs all 10 examples with GPU acceleration
4. Runs brain clot + breast tumor MIMO simulations
5. Saves ALL results + plots to `/kaggle/working/` (persistent)
6. Commits everything to GitHub automatically

**Setup:** Add your GitHub token as a Kaggle Secret:
- Notebook → Add-ons → Secrets → Add New Secret
- Name: `GITHUB_TOKEN`, Value: your PAT (Contents: Read+Write)


In [ ]:
# Check GPU setup
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout)
# Check how many GPUs
r2 = subprocess.run(['nvidia-smi','--query-gpu=index,name,memory.total',
                     '--format=csv,noheader'], capture_output=True, text=True)
gpus = [line.strip() for line in r2.stdout.strip().split('\n') if line.strip()]
print(f'Available GPUs: {len(gpus)}')
for g in gpus: print(f'  {g}')


In [ ]:
# Install + setup
import subprocess, sys, os
subprocess.run(['pip','install','torch','numpy','matplotlib','--quiet'], check=True)

# Clone repo
if not os.path.exists('/kaggle/working/waveforge'):
    subprocess.run(['git','clone','https://github.com/shahzaibshazoo/waveforge.git',
                    '/kaggle/working/waveforge'], check=True)
else:
    subprocess.run(['git','-C','/kaggle/working/waveforge','pull'], check=True)
    print('Repo updated')

sys.path.insert(0, '/kaggle/working/waveforge/src')
os.chdir('/kaggle/working/waveforge')
print('Setup complete.')


In [ ]:
import torch, time, json, re, datetime, shutil
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

N_GPUS   = torch.cuda.device_count()
DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'
print(f'PyTorch: {torch.__version__}')
print(f'GPUs available: {N_GPUS}')
for i in range(N_GPUS):
    name = torch.cuda.get_device_name(i)
    mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f'  GPU {i}: {name} ({mem:.1f} GB)')

# Output directories
RESULTS_DIR = Path('/kaggle/working/results')
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR/'plots').mkdir(exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')


## Part 1: Multi-GPU Throughput Benchmark

In [ ]:
from core import YeeGrid, FieldSet, MurABC, GaussianPulse, PointSource, SourceCollection, FDTD2D

GRID_SIZES = [64, 128, 256, 512, 1024]
N_WARMUP   = 30
N_STEPS    = 300
DX         = 1e-3

def bench_grid(N, device):
    try:
        grid     = YeeGrid(N, N, dx=DX, dy=DX, device=device)
        fields   = FieldSet(grid)
        boundary = MurABC(grid, fields.Hz)
        pulse    = GaussianPulse(1.0, sigma=30*grid.dt)
        src      = PointSource(pulse, N//2, N//2, 'Hz', grid=grid,
                               N_steps=N_WARMUP+N_STEPS)
        sim      = FDTD2D(grid, fields, boundary, SourceCollection([src]), n_check=1000)
        sim.run(N_WARMUP)
        if 'cuda' in device: torch.cuda.synchronize()
        t0 = time.perf_counter()
        sim.run(N_STEPS)
        if 'cuda' in device: torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
        return N_STEPS * N * N / elapsed / 1e6, elapsed / N_STEPS * 1000
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            return None, None  # OOM
        raise

all_gpu_results = {}  # device → list of results

# Benchmark each GPU separately
devices_to_bench = [f'cuda:{i}' for i in range(N_GPUS)] if N_GPUS > 1 else (['cuda'] if DEVICE=='cuda' else ['cpu'])

for dev in devices_to_bench:
    dev_name = torch.cuda.get_device_name(int(dev.split(':')[-1])) if 'cuda' in dev else 'CPU'
    print(f'\n--- {dev_name} ({dev}) ---')
    gpu_rows = []
    for N in GRID_SIZES:
        m, ms = bench_grid(N, dev)
        if m is None:
            print(f'  {N:5d}²  OOM (grid too large for VRAM)')
            break
        gpu_rows.append({'N':N,'mcells_s':round(m,1),'ms_step':round(ms,3)})
        print(f'  {N:5d}²  {m:8.1f} Mcells/s  {ms:7.3f} ms/step')
    all_gpu_results[dev] = {'name': dev_name, 'rows': gpu_rows}

print('\nBenchmark complete.')


## Part 2: Run All 10 Examples

In [ ]:
EXAMPLES = [
    ('01_basic_wave',       'examples/basic_2d_wave.py',            '128x128 free-space',  128, 128,  500),
    ('02_dielectric_slab',  'examples/02_dielectric_slab.py',       '200x64  slab R/T',    200,  64,  800),
    ('03_waveguide',        'examples/03_waveguide.py',             '200x60  waveguide',   200,  60, 1000),
    ('04_scattering',       'examples/04_scattering_cylinder.py',   '150x150 cylinder',    150, 150,  600),
    ('05_through_wall',     'examples/05_through_wall_radar.py',    '200x100 through-wall',200, 100,  600),
    ('06_interference',     'examples/06_multipath_interference.py','128x128 WiFi 2.4GHz', 128, 128,  800),
    ('07_tissue',           'examples/07_tissue_penetration.py',    '250x80  tissue',      250,  80,  800),
    ('08_ev_radar',         'examples/08_ev_radar_ula.py',          '200x200 EV radar',    200, 200,  600),
    ('09_brain_clot',       'examples/09_brain_clot_dataset_sample.py','150x150 brain MIMO',150,150,12800),
    ('10_breast_tumor',     'examples/10_breast_tumor_mimo.py',     '200x200 breast MIMO', 200, 200,16000),
]

example_results = []
PYTHON = sys.executable

for name, script, desc, NX, NY, total_steps in EXAMPLES:
    print(f'\n[{name}] {desc}...')
    t0  = time.perf_counter()
    r   = subprocess.run([PYTHON, script], capture_output=True, text=True, timeout=1200)
    elapsed = time.perf_counter() - t0

    # Parse throughput
    bench = re.findall(r'WAVEFORGE_BENCH:\s*([\d]+\.?[\d]*)', r.stdout)
    if bench:
        mcells = max(float(v) for v in bench)
    else:
        vals = re.findall(r'([\d]+\.?[\d]*)\s*Mcells/s', r.stdout)
        mcells = max((float(v) for v in vals), default=0.0)
    if mcells == 0.0:
        mcells = round(total_steps * NX * NY / max(elapsed,1e-9) / 1e6, 1)

    status = 'OK' if r.returncode == 0 else 'FAIL'
    example_results.append({'name':name,'desc':desc,'status':status,
                             'time_s':round(elapsed,1),'mcells_s':round(mcells,1)})
    print(f'  {status} | {elapsed:.1f}s | {mcells:.1f} Mcells/s')
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-500:])

# Copy output plots to results dir
import glob
for p in glob.glob('examples/output/*.png'):
    shutil.copy(p, RESULTS_DIR/'plots'/Path(p).name)
    # Also update docs/assets for website
    shutil.copy(p, f'docs/assets/simulations/{Path(p).name}')
print(f'\nPlots saved to {RESULTS_DIR}/plots/')


## Part 3: Multi-GPU Speedup Chart

In [ ]:
# Build comparison chart
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#070b14')
for ax in axes:
    ax.set_facecolor('#0d1424')
    ax.tick_params(colors='#8892a4')
    for sp in ax.spines.values(): sp.set_edgecolor('#1a2540')

fig.suptitle(f'WaveForge — Multi-GPU Benchmark on Kaggle ({GPU_NAME})\n'
             f'{N_GPUS} GPU(s) · {datetime.datetime.now().strftime("%Y-%m-%d")}',
             fontsize=13, fontweight='bold', color='#f0f6ff')

gpu_palette = ['#4fd1c5','#63b3ed','#9f7aea','#f0b429']

# Left: throughput scaling per GPU
ax = axes[0]
for di, (dev, info) in enumerate(all_gpu_results.items()):
    rows = info['rows']
    if rows:
        Ns = [r['N'] for r in rows]
        ms = [r['mcells_s'] for r in rows]
        ax.plot(Ns, ms, 'o-', lw=2.5, ms=8, color=gpu_palette[di],
                label=f"{info['name']} ({dev})")
        # Annotate peak
        ax.annotate(f'{max(ms):.0f}',
                    (Ns[ms.index(max(ms))], max(ms)),
                    textcoords='offset points', xytext=(4,6),
                    color=gpu_palette[di], fontsize=9, fontweight='bold')

# Add Meep CPU reference
meep_ref = [14.2, 20.3, 15.3, 16.1]
ax.plot([64,128,256,512], meep_ref, '^--', lw=1.5, ms=7,
        color='#fc8181', alpha=0.7, label='Meep CPU (laptop)')
ax.set(xlabel='Grid size (N×N)', ylabel='Throughput (Mcells/s)',
       title='GPU Throughput Scaling')
valid_Ns = [r['N'] for r in list(all_gpu_results.values())[0]['rows']]
ax.set_xticks(valid_Ns); ax.set_xticklabels([f'{N}²' for N in valid_Ns])
ax.legend(fontsize=8, facecolor='#0d1424', labelcolor='#8892a4', edgecolor='#1a2540')
ax.grid(True, alpha=0.15, color='#1a2540')

# Right: examples throughput
ax2 = axes[1]
names  = [r['name'].replace('_',' ') for r in example_results]
mcells = [r['mcells_s'] for r in example_results]
colors = ['#2ca02c' if r['status']=='OK' else '#d62728' for r in example_results]
bars = ax2.barh(names, mcells, color=colors, alpha=0.9, height=0.6)
ax2.bar_label(bars, [f'{v:.0f}' for v in mcells], fontsize=9,
              padding=3, color='#f0f6ff', fontweight='600')
ax2.set(xlabel='Mcells/s', title='All Examples GPU Throughput')
ax2.grid(True, alpha=0.15, axis='x', color='#1a2540')

plt.tight_layout()
chart_p = RESULTS_DIR/'kaggle_gpu_benchmark.png'
fig.savefig(chart_p, dpi=150, bbox_inches='tight', facecolor='#070b14')
shutil.copy(chart_p, 'assets/kaggle_gpu_benchmark.png')
shutil.copy(chart_p, 'docs/assets/kaggle_gpu_benchmark.png')
plt.show()
print(f'Chart saved: {chart_p}')


## Part 4: Save All Results (Persistent)

In [ ]:
# Collect all results into master JSON
master = {
    'meta': {
        'date':     datetime.datetime.now().isoformat(),
        'platform': 'Kaggle',
        'n_gpus':   N_GPUS,
        'gpu_name': GPU_NAME,
        'torch':    torch.__version__,
        'measured': True,
    },
    'gpu_scaling':   all_gpu_results,
    'examples':      example_results,
    'summary': {
        'examples_passed': sum(1 for r in example_results if r['status']=='OK'),
        'examples_total':  len(example_results),
        'peak_mcells_s':   max((r['mcells_s'] for r in example_results), default=0),
        'total_example_time_s': sum(r['time_s'] for r in example_results),
    },
}

# Save to persistent Kaggle storage AND repo
for path in [
    RESULTS_DIR/'kaggle_results.json',
    Path('benchmarks/kaggle_gpu_results.json'),
]:
    with open(path, 'w') as f:
        json.dump(master, f, indent=2)
print(f'Results saved to {RESULTS_DIR}/')
print(f'All plots in {RESULTS_DIR}/plots/')
print(f'Summary: {master["summary"]}')

# List everything saved
print('\nFiles in results dir:')
for p in sorted(RESULTS_DIR.rglob('*')):
    if p.is_file(): print(f'  {p.relative_to(RESULTS_DIR)}')


## Part 5: Commit Everything to GitHub

In [ ]:
# Load GitHub token from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_TOKEN = UserSecretsClient().get_secret('GITHUB_TOKEN')
    print('Token loaded from Kaggle Secrets')
except Exception as e:
    GITHUB_TOKEN = ''
    print(f'Could not load secret ({e})')
    print('Add GITHUB_TOKEN via: Notebook → Add-ons → Secrets')

import subprocess as sp
os.chdir('/kaggle/working/waveforge')
sp.run(['git','config','user.name','Shahzaib Ur Rehman'])
sp.run(['git','config','user.email','shahzaib@waveforge.io'])

if GITHUB_TOKEN:
    sp.run(['git','remote','set-url','origin',
            f'https://{GITHUB_TOKEN}@github.com/shahzaibshazoo/waveforge.git'])
    sp.run(['git','pull','--rebase','origin','main'],
           capture_output=True)  # sync first

    # Stage all new files
    sp.run(['git','add',
            'benchmarks/kaggle_gpu_results.json',
            'assets/kaggle_gpu_benchmark.png',
            'docs/assets/kaggle_gpu_benchmark.png',
            'docs/assets/simulations/',
            'docs/assets/all_examples_gpu_benchmark.png',
    ])

    n_gpus_str = f'{N_GPUS}x {GPU_NAME}'
    date_str   = datetime.datetime.now().strftime('%Y-%m-%d')
    commit_msg = f'Kaggle benchmark: {n_gpus_str} — all examples — {date_str}'

    r = sp.run(['git','commit','-m', commit_msg], capture_output=True, text=True)
    if 'nothing to commit' in r.stdout:
        print('Nothing new to commit (already up to date)')
    else:
        push = sp.run(['git','push','origin','main'], capture_output=True, text=True)
        if push.returncode == 0:
            print(f'Pushed to GitHub!')
            print(f'  Benchmarks: https://github.com/shahzaibshazoo/waveforge/tree/main/benchmarks')
            print(f'  Website:    https://shahzaibshazoo.github.io/waveforge')
        else:
            print('Push failed:', push.stderr[-500:])
else:
    print('Files saved locally at /kaggle/working/results/')
    print('Download them from the Output panel on the right →')


## Part 6: Final Summary

In [ ]:
from IPython.display import Image, display

print('='*65)
print(f'  WaveForge — Kaggle Benchmark Complete')
print(f'  GPU: {GPU_NAME} × {N_GPUS}')
print(f'  Date: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
print('='*65)

# GPU scaling table
print('\nGPU Throughput Scaling:')
for dev, info in all_gpu_results.items():
    print(f'  {info["name"]} ({dev})')
    for r in info['rows']:
        peak_marker = ' ← PEAK' if r['mcells_s'] == max(row['mcells_s'] for row in info['rows']) else ''
        print(f'    {r["N"]:5d}²  {r["mcells_s"]:8.1f} Mcells/s{peak_marker}')

print('\nExamples:')
ok = sum(1 for r in example_results if r['status']=='OK')
for r in example_results:
    print(f'  {r["name"]:<28} {r["status"]:>4}  {r["mcells_s"]:>7.1f} Mcells/s  {r["time_s"]:>6.1f}s')
print(f'\n  Passed: {ok}/{len(example_results)}')
print('='*65)

display(Image(str(RESULTS_DIR/'kaggle_gpu_benchmark.png')))
